# PHIL-TEXT — Faz 4.3: Hiperparametre Optimizasyonu (Optuna)

**Amaç:** LinearSVC ve LogisticRegression için Optuna TPE sampler ile otomatik hiperparametre arama.

### Adımlar
1. Kurulum & Veri Yükleme  
2. LinearSVC Optuna Study  
3. LogisticRegression Optuna Study  
4. Optimizasyon Görselleştirmeleri  
5. Final Model + Test Değerlendirmesi  
6. Model Kaydetme  
7. Özet Rapor

## 1. Kurulum & Veri Yükleme

In [ ]:
import sys, os, json, time, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

try:
    import optuna
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f'Optuna: {optuna.__version__}')
except ImportError:
    raise ImportError('Optuna yuklu degil. Yuklemek icin: pip install optuna')

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

from src.models.evaluate import evaluate_classification, plot_confusion_matrix

plt.rcParams['figure.dpi'] = 130
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
data = np.load('../data/processed/splits.npz', allow_pickle=True)
X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']

with open('../data/processed/label_mappings.json', encoding='utf-8') as f:
    mappings = json.load(f)
id2label = {v: k for k, v in mappings['philosopher'].items()}

# TF-IDF (bir kez fit, her model paylaşır)
TFIDF = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2),
                         sublinear_tf=True, stop_words='english')
X_train_tfidf = TFIDF.fit_transform(X_train)
X_val_tfidf   = TFIDF.transform(X_val)
X_test_tfidf  = TFIDF.transform(X_test)

X_trainval = np.concatenate([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'TF-IDF shape: {X_train_tfidf.shape}')

## 2. LinearSVC Optuna Study

In [ ]:
def objective_linearsvc(trial):
    C     = trial.suggest_float('C', 0.01, 100.0, log=True)
    max_i = trial.suggest_int('max_iter', 1000, 5000)
    loss  = trial.suggest_categorical('loss', ['hinge', 'squared_hinge'])
    clf   = LinearSVC(C=C, max_iter=max_i, loss=loss,
                      class_weight='balanced', random_state=42)
    skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X_train_tfidf, y_train,
                              cv=skf, scoring='f1_weighted', n_jobs=-1)
    return scores.mean()

print('LinearSVC Optuna study (50 deneme)...')
t0 = time.time()
study_svc = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_svc.optimize(objective_linearsvc, n_trials=50, show_progress_bar=False)

print(f'Sure       : {time.time()-t0:.1f}s')
print(f'Best F1 CV : {study_svc.best_value:.4f}')
print(f'Best params: {study_svc.best_params}')

## 3. LogisticRegression Optuna Study

In [ ]:
def objective_lr(trial):
    C       = trial.suggest_float('C', 0.001, 100.0, log=True)
    solver  = trial.suggest_categorical('solver', ['lbfgs', 'saga'])
    penalty = 'l2'
    if solver == 'saga':
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    clf = LogisticRegression(C=C, solver=solver, penalty=penalty,
                              class_weight='balanced', max_iter=2000,
                              random_state=42, n_jobs=-1)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X_train_tfidf, y_train,
                              cv=skf, scoring='f1_weighted', n_jobs=-1)
    return scores.mean()

print('LogisticRegression Optuna study (50 deneme)...')
t0 = time.time()
study_lr = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_lr.optimize(objective_lr, n_trials=50, show_progress_bar=False)

print(f'Sure       : {time.time()-t0:.1f}s')
print(f'Best F1 CV : {study_lr.best_value:.4f}')
print(f'Best params: {study_lr.best_params}')

## 4. Optimizasyon Görselleştirmeleri

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

svc_vals = [t.value for t in study_svc.trials if t.value is not None]
lr_vals  = [t.value for t in study_lr.trials  if t.value is not None]

for ax, vals, title, color in [
    (axes[0], svc_vals, 'LinearSVC',         'steelblue'),
    (axes[1], lr_vals,  'LogisticRegression', 'darkorange'),
]:
    ax.plot(vals, alpha=0.4, color=color, linewidth=1, label='Deneme F1')
    ax.plot(np.maximum.accumulate(vals), color='red',
            linewidth=2, label='Kumulatif En Iyi')
    ax.set_xlabel('Deneme No')
    ax.set_ylabel('F1 Weighted (5-Fold CV)')
    ax.set_title(f'{title} — Optuna Gecmisi')
    ax.legend()
    ax.set_ylim(0.85, 1.01)

plt.suptitle('Faz 4.3 — Hiperparametre Optimizasyonu', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/optuna_history.png', bbox_inches='tight')
plt.show()

In [ ]:
# Parametre önem grafikleri
try:
    from optuna.visualization.matplotlib import plot_param_importances
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_param_importances(study_svc, ax=axes[0])
    axes[0].set_title('LinearSVC — Parametre Onem')
    plot_param_importances(study_lr, ax=axes[1])
    axes[1].set_title('LogisticReg — Parametre Onem')
    plt.tight_layout()
    plt.savefig('../docs/optuna_param_importance.png', bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Param importance grafik cizilemiyor: {e}')

## 5. Final Model + Test Değerlendirmesi

In [ ]:
svc_p = study_svc.best_params
best_svc = LinearSVC(C=svc_p['C'], max_iter=svc_p['max_iter'],
                      loss=svc_p['loss'], class_weight='balanced', random_state=42)

# Val ile degerlendirme
best_svc.fit(X_train_tfidf, y_train)
val_pred    = best_svc.predict(X_val_tfidf)
val_metrics = evaluate_classification(y_val, val_pred, id2label)
print(f'Val F1 (W): {val_metrics["f1_weighted"]:.4f}')

# Test seti — trainval ile yeniden egit
X_tv_tfidf = TFIDF.transform(X_trainval)
best_svc.fit(X_tv_tfidf, y_trainval)
test_pred    = best_svc.predict(X_test_tfidf)
test_metrics = evaluate_classification(y_test, test_pred, id2label)

print(f'Test Acc   : {test_metrics["accuracy"]:.4f}')
print(f'Test F1 (W): {test_metrics["f1_weighted"]:.4f}')
print(f'Test F1 (M): {test_metrics["f1_macro"]:.4f}')
print()
print(test_metrics['report'])

In [ ]:
cm = plot_confusion_matrix(y_test, test_pred, id2label,
                            save_path='../docs/optuna_confusion_matrix.png')

## 6. Model Kaydetme

In [ ]:
os.makedirs('../models', exist_ok=True)

# Tam pipeline (tfidf + clf) — deploy icin hazir
final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10_000, ngram_range=(1, 2),
                               sublinear_tf=True, stop_words='english')),
    ('clf',   LinearSVC(C=svc_p['C'], max_iter=svc_p['max_iter'],
                         loss=svc_p['loss'], class_weight='balanced', random_state=42)),
])
final_pipeline.fit(X_trainval, y_trainval)
joblib.dump(final_pipeline, '../models/best_model_optuna_linearsvc.joblib')

optuna_meta = {
    'model': 'LinearSVC_Optuna',
    'best_params':      svc_p,
    'val_f1_weighted':  round(val_metrics['f1_weighted'],  4),
    'test_accuracy':    round(test_metrics['accuracy'],    4),
    'test_f1_weighted': round(test_metrics['f1_weighted'], 4),
    'baseline_test_f1': 0.9968,
    'n_trials': 50,
}
with open('../models/optuna_best_params.json', 'w') as f:
    json.dump(optuna_meta, f, indent=2)

print('Kaydedildi: models/best_model_optuna_linearsvc.joblib')
print('Kaydedildi: models/optuna_best_params.json')

## 7. Özet Rapor

In [ ]:
print('=' * 60)
print('  FAZ 4.3 — HIPERPARAMETRE OPTIMIZASYONU OZET')
print('=' * 60)
print(f'  Baseline LinearSVC   F1(W)=0.9968  (sabit params)')
print(f'  Optuna LinearSVC     F1(W)={val_metrics["f1_weighted"]:.4f}  (optimize)')
print(f'  Optuna LR (val)      F1(W)={study_lr.best_value:.4f}')
print()
print(f'  Best SVC params : {svc_p}')
print()
print('  Sonraki: Faz 4.4 — DistilBERT Fine-tuning')
print('=' * 60)